# ============================================================
# EDA PLOTS(Exploratory Data Analysis plots) (TUMC, 5-class + binary)
# ============================================================
# Modernized from the binary/url-only version:
#   - reads the URL column (full URLs)
#   - 5-class aware (benign/phishing/malware/scam/other)
#   - also produces binary benign-vs-malicious views
#   - reads dataset_with_folds_v9/v10.csv directly
#
# Produces 6 publication figures for the dataset chapter.
# 100% offline, fast (~2-3 min).
# ============================================================

In [ ]:
import os, re, math, warnings
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tldextract

warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
# Prefer the feature file (has url + class_final); fall back to raw dataset
for cand in ["dataset_with_folds_v10.csv", "dataset_with_folds_v9.csv",
             "dataset_with_folds_v10.csv"]:
    INPUT_FILE = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT_FILE):
        break
OUT_DIR = os.path.join(DATA_DIR, "eda_plots")
os.makedirs(OUT_DIR, exist_ok=True)

# Consistent 5-class palette
C5 = {"benign":"#1D9E75","phishing":"#D85A30","malware":"#9B2226",
      "scam":"#BC6C25","other_malicious":"#5B7BB8"}
C2 = {"benign":"#1D9E75","malicious":"#D85A30"}

def url_entropy(s):
    s = str(s)
    if not s: return 0.0
    freq = Counter(s); n = len(s)
    return -sum((c/n)*math.log2(c/n) for c in freq.values())

print("="*70)
print("STEP 1 — EDA PLOTS (TUMC)")
print("="*70)
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"  Loaded {len(df):,} rows from {os.path.basename(INPUT_FILE)}")

# Derive plotting columns if absent
if "url_len" not in df.columns:
    df["url_len"] = df["url"].astype(str).str.len()
if "url_entropy" not in df.columns:
    df["ent_plot"] = df["url"].astype(str).apply(url_entropy)
else:
    df["ent_plot"] = df["url_entropy"]
if "num_subdomains" not in df.columns:
    df["num_subs_plot"] = df["url"].astype(str).apply(
        lambda u: len([s for s in tldextract.extract(
            u if u.startswith("http") else "http://"+u).subdomain.split(".") if s]))
else:
    df["num_subs_plot"] = df["num_subdomains"]

classes = ["benign","phishing","malware","scam","other_malicious"]
present = [c for c in classes if c in df["class_final"].unique()]

# ── Plot 1 — URL length (5-class) ────────────────────────────
fig, (a1,a2) = plt.subplots(1,2,figsize=(14,4.5))
for cls in present:
    g = df[df["class_final"]==cls]
    a1.hist(g["url_len"].clip(upper=200), bins=60, alpha=0.55,
            label=cls, color=C5[cls], density=True)
a1.set(xlabel="URL length (clipped 200)", ylabel="Density",
       title="URL length by class (5-class)")
a1.legend(fontsize=8)
for lbl in ["benign","malicious"]:
    g = df[df["label"]==lbl]
    a2.hist(g["url_len"].clip(upper=200), bins=60, alpha=0.6,
            label=lbl, color=C2[lbl], density=True)
a2.set(xlabel="URL length (clipped 200)", ylabel="Density",
       title="URL length (binary)")
a2.legend()
plt.suptitle("TUMC — URL Length Distribution", fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda1_url_length.png",dpi=150); plt.close()
print("  Saved eda1_url_length.png")

# ── Plot 2 — Entropy ─────────────────────────────────────────
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,4.5))
for cls in present:
    g=df[df["class_final"]==cls]
    a1.hist(g["ent_plot"],bins=60,alpha=0.55,label=cls,color=C5[cls],density=True)
a1.set(xlabel="Shannon entropy",ylabel="Density",title="URL entropy by class")
a1.legend(fontsize=8)
for lbl in ["benign","malicious"]:
    g=df[df["label"]==lbl]
    a2.hist(g["ent_plot"],bins=60,alpha=0.6,label=lbl,color=C2[lbl],density=True)
a2.set(xlabel="Shannon entropy",ylabel="Density",title="URL entropy (binary)")
a2.legend()
plt.suptitle("TUMC — URL Entropy Distribution",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda2_entropy.png",dpi=150); plt.close()
print("  Saved eda2_entropy.png")

# ── Plot 3 — Class distribution ──────────────────────────────
fig,(a1,a2)=plt.subplots(1,2,figsize=(13,4.5))
counts = df["class_final"].value_counts().reindex(present)
a1.bar(range(len(counts)),counts.values,
       color=[C5[c] for c in counts.index],alpha=0.85)
a1.set_xticks(range(len(counts))); a1.set_xticklabels(counts.index,rotation=30,ha="right")
a1.set(ylabel="Count",title="Five-class composition")
for i,v in enumerate(counts.values):
    a1.text(i,v,f"{v:,}\n({v/len(df)*100:.1f}%)",ha="center",va="bottom",fontsize=8)
binc = df["label"].value_counts()
a2.bar(range(len(binc)),binc.values,color=[C2[c] for c in binc.index],alpha=0.85)
a2.set_xticks(range(len(binc))); a2.set_xticklabels(binc.index)
a2.set(ylabel="Count",title=f"Binary composition ({binc.get('benign',0)/binc.get('malicious',1):.2f}:1)")
for i,v in enumerate(binc.values):
    a2.text(i,v,f"{v:,}",ha="center",va="bottom",fontsize=9)
plt.suptitle("TUMC — Class Composition",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda3_class_dist.png",dpi=150); plt.close()
print("  Saved eda3_class_dist.png")

# ── Plot 4 — Top 20 TLDs ─────────────────────────────────────
top = (df.groupby(["tld","label"]).size().unstack(fill_value=0)
       .assign(total=lambda x:x.sum(axis=1)).nlargest(20,"total").drop(columns="total"))
for col in ["benign","malicious"]:
    if col not in top.columns: top[col]=0
fig,ax=plt.subplots(figsize=(11,5))
top[["benign","malicious"]].plot(kind="bar",ax=ax,
    color=[C2["benign"],C2["malicious"]],width=0.75,edgecolor="none")
ax.set(xlabel="TLD",ylabel="Count",title="Top 20 TLDs — benign vs malicious")
ax.tick_params(axis="x",rotation=45); ax.legend(["Benign","Malicious"])
plt.suptitle("TUMC — TLD Distribution",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda4_top_tlds.png",dpi=150); plt.close()
print("  Saved eda4_top_tlds.png")

# ── Plot 5 — Subdomain depth ─────────────────────────────────
fig,ax=plt.subplots(figsize=(9,4.5))
for cls in present:
    g=df[df["class_final"]==cls]
    c=g["num_subs_plot"].value_counts().sort_index()
    ax.plot(c.index,c.values/c.sum(),marker="o",label=cls,color=C5[cls])
ax.set(xlabel="Subdomain depth",ylabel="Proportion",xlim=(-0.2,8),
       title="Subdomain depth by class")
ax.legend(fontsize=8)
plt.suptitle("TUMC — Subdomain Depth",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda5_subdomain_depth.png",dpi=150); plt.close()
print("  Saved eda5_subdomain_depth.png")

# ── Plot 6 — Turkish keyword presence by class ───────────────
TR_KW = [    "giris","giriş","oturum","uye-giris","uyegiris",
    "giris-yap","girisyap","hesabim","hesabım",
    "hesap-dogrula","sifre","şifre","sifre-sifirla",
    "sifre-yenile","parola","kullanici","kullanıcı",
    "uyelik","üyelik",

    # Ödeme / bankacılık
    "odeme","ödeme","banka","bankaci","bankacilik","bankacılık",
    "mobil-banka","internet-bankaciligi","kart-bilgi",
    "kart-dogrulama","iban","swift","havale","eft",
    "para-transferi","para-gonder","kredi-karti","kredikarti",
    "debit-karti","banka-hesabi","hesap-bakiye",

    # Devlet servisleri
    "e-devlet","edevlet","e-nabiz","enabiz","e-okul",
    "mhrs","sgk","vergi-dairesi","gib","nvi",
    "tckimlik","tc-kimlik","tc-no","tc-numarasi",
    "kimlik-dogrulama","pasaport","ehliyet",

    # Kargo / lojistik
    "kargo-takip","kargom","teslimat","gonderitakibi",
    "siparis","sipariş","siparis-takip","fatura",
    "aras-kargo","araskargo","yurtici-kargo",
    "mng-kargo","surat-kargo","ups-kargo",

    # Doğrulama / güvenlik
    "dogrula","doğrula","dogrulama","doğrulama",
    "kimlik-dogrula","guvenli","güvenli",
    "guvenlik","güvenlik","guvenlik-kodu",
    "onay","onayla","onaylama",
    "sms-dogrulama","otp-dogrulama",

    # Türk banka markaları
    "garanti-bbva","garantibbva","garanti-bankasi",
    "akbank","isbank","isbankasi","is-bankasi",
    "ziraat-bankasi","ziraatbankasi",
    "halk-bank","halkbank","vakif-bank","vakifbank",
    "yapi-kredi","yapikredi","deniz-bank","denizbank",
    "qnb-finansbank","teb-bankasi","enpara",
    "papara","paycell","tosla",
    "paribu","btcturk","binance-tr",

    # E-ticaret markaları
    "trendyol","hepsiburada","n11",
    "ciceksepeti","gittigidiyor","sahibinden",

    # Telekom markaları
    "turkcell","vodafone-tr","turktelekom","ttnet","bimcell",

    # URL yapısal sinyalleri
    "musteri-hizmetleri","musteri-destek",
    "destek-hatti","islem","musteri",
    "basvuru","kampanya","hesap-giris","kredi","iade",

    # İlan / emlak oltalama
    "sahibinden-giris","letgo-giris","emlak-giris",

    # Hediye / kampanya
    "hediye-kazan","hediyekampanya","hediyeniz",]
rates = {}
for cls in present:
    u = df[df["class_final"]==cls]["url"].astype(str).str.lower()
    rates[cls] = {kw: u.str.contains(kw,na=False).mean()*100 for kw in TR_KW}
kw_df = pd.DataFrame(rates)
kw_df["spread"] = kw_df.max(axis=1)-kw_df.min(axis=1)
kw_df = kw_df.sort_values("spread",ascending=True).drop(columns="spread").tail(18)
fig,ax=plt.subplots(figsize=(10,7))
yy=range(len(kw_df))
for cls in present:
    if cls in kw_df.columns:
        ax.barh(yy,kw_df[cls],alpha=0.6,label=cls,color=C5[cls])
ax.set_yticks(list(yy)); ax.set_yticklabels(kw_df.index)
ax.set_xlabel("% of URLs containing keyword")
ax.set_title("Turkish keyword presence by class")
ax.legend(fontsize=8)
plt.suptitle("TUMC — Turkish Keyword Distribution",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda6_turkish_keywords.png",dpi=150); plt.close()
print("  Saved eda6_turkish_keywords.png")

print(f"\n  All 6 EDA plots → {OUT_DIR}/")
print("="*70)